<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-architect-program/blob/main/Day_20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#MCP Hands-On: Build One Server, Run It on a **Local Model** (LM Studio) and on Claude

**The one-line story of today:** You build **one** MCP server, then plug it into different "brains" — a **free, offline model running in LM Studio on your own laptop** *and* Claude — **without changing a single line of the server**. That "build once, run anywhere" moment is the whole point of MCP, and today you feel it in your hands.

---


### 🎯 What you'll be able to do by the end
- **Explain** MCP end to end: host, client, server, the 6 primitives, transports, and the message loop — in plain words.
- **Build** a real MCP server with FastMCP that exposes a **tool**, a **resource**, and a **prompt**.
- **Run it on a local open-source model** in **LM Studio** — no API key, no cloud — and also on Claude, unchanged.
- **Call MCP from LM Studio's API** (ephemeral + `mcp.json`), so your code can drive a local model + tools.

### 🧭 The rhythm of each section
| Block | Meaning |
|---|---|
| 🧠 The idea | The concept in plain English |
| 💻 Do this | A command or code cell you actually run |
| 🤯 Fun fact | One true, checked detail to make it stick |
| 🎤 Interview angle | How this shows up in a real interview |
| ✅ Quick check | A tiny question with the answer right there |


### 🏬 Today's scenario
You're the first AI engineer at **"QuickCart"**, a Zepto-style 10-minute-delivery startup. Ops keeps messaging you: *"total orders per city?"*, *"export today's orders as CSV"*. Order data is sensitive (customer PII), so it **must stay on-premise** — nothing to the cloud. You'll wrap QuickCart's orders database in **one** MCP server, then let a **private, offline model in LM Studio** answer those questions safely. The same server also works on Claude for non-sensitive use.

---

# Foundations — The 5 Words You Need (60-second recap)

Before we build, lock in the vocabulary. The MCP "shape" has five moving parts:

| Word | Plain meaning | In today's lab |
|---|---|---|
| **Host** | The AI *app* the human talks to | **LM Studio** (primary), Claude Desktop, Claude Code |
| **Client** | The little connector *inside* the host — one per server | Created automatically when you register a server |
| **Server** | The program *you* write that exposes capabilities | `server.py` (QuickCart orders) |
| **Tool** | An *action* the AI can call | `query_orders(sql)` |
| **Resource** | *Read-only data* the AI can pull in | `orders://export/csv` |

🧠 **The idea in one sentence:** MCP is a **USB-C port for AI apps** — you build the "device" (server) once, and it plugs into any "laptop" (host) that has the port, no matter who made the laptop.

> **Accuracy note:** MCP is a *protocol*, not a product. Anthropic created it, but it is **model-agnostic and vendor-neutral** — the same server works with a local Qwen model, Claude, GPT, or Gemini. As of **December 2025**, MCP is no longer owned by Anthropic at all: it was **donated to the Linux Foundation** (see the next section). An interviewer loves when you say *"MCP isn't a Claude feature — it's an open standard that Claude, LM Studio, and everyone else speaks."*

🎤 **Interview angle:** *"Why is MCP a big deal for enterprises?"* -> "It turns AI integrations from **N×M** custom glue (every app × every tool) into **N+M** — build a tool once as a server, and every AI app can reuse it. That's the difference between a science project and infrastructure."

<details><summary>✅ Quick check — Host vs Server?</summary>

**Host** = the AI app (the "laptop") — LM Studio in this lab. **Server** = the program you wrote that offers tools/resources (the "USB device"). The **client** is the port inside the laptop that the device plugs into.
</details>

---

# Section 1 — What MCP *Really* Is (under the hood)

You've used the "USB-C" picture. Now let's open the box, because to handle MCP **without any guidance** you need to know exactly what's inside. This is the part most tutorials skip — and the part interviews dig into.

### What is it?
MCP (**Model Context Protocol**) is an **open standard** for how an AI app (host) and a capability provider (server) talk to each other. Under the "USB-C" metaphor is a very ordinary mechanism: **JSON-RPC 2.0 messages** passed back and forth. JSON-RPC is just a simple format for "call this function with these arguments, here's the result" — the same idea as calling an API, written as small JSON messages.

### Why does it matter?
Before MCP, every AI app integrated every tool its **own** way. Ten apps × ten tools = a hundred custom integrations. MCP defines **one** way to describe tools and data, so any host that speaks MCP can use any server that speaks MCP. Build once, reuse everywhere.

### How does it work? (the message loop, step by step)
1. **The host launches the server** and they do a **handshake** (called *initialization*). They exchange versions and **capabilities** — "I can offer tools and resources", "I can accept those".
2. The host asks **"what have you got?"** -> `tools/list`, `resources/list`, `prompts/list`. The server answers with names, descriptions, and JSON schemas.
3. When the model wants an action, the host sends **`tools/call`** with the tool name + arguments. The server runs it and returns the result.
4. The result goes back to the model, which writes the final answer.

That's it. Everything fancy in MCP is built on this request -> run -> result loop.

## 🧩 The 6 primitives (the full MCP surface)

A "primitive" is a *type of thing* a server or host can offer. There are **six**, split by who provides them. You'll mostly use the first three today.

**Server-side (the server offers these to the AI):**

| Primitive | Plain meaning | QuickCart example |
|---|---|---|
| **Tool** 🔧 | An **action** the model can call (it *does* something) | `query_orders(sql)` runs a SELECT |
| **Resource** 📄 | **Read-only data/context** the model can pull in (it *reads* something) | `orders://export/csv` returns the orders as CSV |
| **Prompt** 💬 | A **reusable prompt template** the server offers the user | a `/daily_report` template that fills in today's date |

**Client-side (the host/client offers these back to the server — advanced):**

| Primitive | Plain meaning | When it's used |
|---|---|---|
| **Sampling** 🔁 | The server asks the *host* to run an LLM completion for it (reverse direction!) | A server wants to summarize data without shipping its own model. **The host keeps control of the model + API key.** |
| **Roots** 📁 | The host tells the server which files/folders it's allowed to touch | Scoping a filesystem server to one project directory |
| **Elicitation** 🙋 | The server pauses to **ask the user** for more info mid-task | "Which city did you mean — Chennai or Chandigarh?" |

> 🧠 **Remember this:** *Tool = an action (it runs). Resource = data (it reads). Prompt = a reusable template. The other three (sampling, roots, elicitation) flow the other way and keep the human/host in control.*

❌ **Don't mix these up:**
- ❌ "A resource is just another kind of tool." -> ✅ A **tool runs code and can change things**; a **resource is read-only data** the model pulls in for context (closer to RAG retrieval).
- ❌ "Sampling means the server has its own model." -> ✅ **The opposite** — the server borrows the *host's* model, so the host controls cost, choice, and keys.

## 🚚 Transports — how the messages actually travel

The JSON-RPC messages need a pipe to travel through. MCP defines **two** standard transports:

| Transport | Where the server runs | How it talks | Use it for |
|---|---|---|---|
| **stdio** | On **your machine**, as a local process | Reads JSON from stdin, writes JSON to stdout | **Local servers** — exactly what we build today (LM Studio + Claude launch `python server.py`) |
| **Streamable HTTP** | **Remotely**, as a web service | HTTP POST/GET, can stream results with Server-Sent Events | **Remote/hosted servers** (e.g. Hugging Face's public MCP server) |


💻 **Today we use `stdio`** because the server runs on your laptop next to your private data — nothing leaves the machine. Later in Part B you'll also point LM Studio at a **remote** Streamable-HTTP server (Hugging Face) to feel the difference.

🤯 **Fun fact:** the whole protocol is built on **JSON-RPC 2.0**, a spec from **2010** — older than most JavaScript frameworks. MCP didn't invent a new wire format; it reused a boring, proven one. Good infrastructure is often "boring tech, new arrangement".

## 🏛️ Where MCP came from (and who owns it now)

This is a favourite interview question because it changed **recently** — most people don't know the update.

- **Nov 2024** — Anthropic **introduces and open-sources** MCP.
- **2025** — Explosive adoption: OpenAI, Google, Microsoft, Cursor, and **LM Studio** all become MCP hosts. It stops being "a Claude thing".
- **9 Dec 2025** — Anthropic **donates MCP to the Linux Foundation**, under the new **Agentic AI Foundation (AAIF)**. Founding orgs include **Anthropic, Block, and OpenAI**; MCP joins two other founding projects — **goose** (Block) and **AGENTS.md** (OpenAI).

> 🧠 **Remember this:** *MCP is now a vendor-neutral standard governed by the Linux Foundation — like how Kubernetes isn't "a Google product" anymore. That neutrality is exactly why an enterprise can adopt it without betting on one AI vendor.*

🎤 **Interview angle:** *"Isn't MCP just Anthropic lock-in?"* -> "The opposite. Anthropic gave it away — since December 2025 it's a Linux Foundation project alongside contributions from OpenAI and Block. Adopting MCP is a bet on a **standard**, not a **vendor**."

<details><summary>✅ Quick check — Which transport for a local server, and what governs MCP today?</summary>

**stdio** for a local server (JSON over stdin/stdout). MCP is governed by the **Linux Foundation's Agentic AI Foundation** since December 2025 — it's vendor-neutral, not owned by Anthropic.
</details>

---

# Section 2 — Feel the Engine: the tool-use loop, live 🔬

🧠 **The idea:** MCP feels like magic, but underneath it's just the **tool-use loop** you met in the Agents weekend: the model *asks* to call a tool -> your code *runs* it -> you *send the result back* -> the model answers. An MCP server is simply a **standard way to package those tools** so any host can use them.

Let's run that exact loop **live** so every config step later makes total sense. We use the Claude API here because it's the quickest way to *see* the loop in code. **The same loop is what LM Studio runs for you automatically in Part B.**

Set your key as a Colab Secret named `MY_API_KEY` first.

In [1]:
# cell 1 — install
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 13.8 MB/s eta 0:00:00


In [2]:
# cell 2 — key + client (Colab Secret named MY_API_KEY)
from google.colab import userdata
import os, sqlite3
os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"   # fast + cheap: the right default for labs
print("Ready ✅")

Ready ✅


In [3]:
# cell 3 — a tiny in-memory orders DB (stands in for quickcart.db)
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE orders (id INTEGER, city TEXT, item TEXT, amount INTEGER)")
db.executemany("INSERT INTO orders VALUES (?,?,?,?)", [
    (1,"Chennai","milk",120),(2,"Chennai","bread",80),
    (3,"Bengaluru","eggs",150),(4,"Bengaluru","milk",90),
])
db.commit()

# This is the SAME read-only guard your MCP tool will use.
def query_orders(sql: str) -> str:
    if not sql.strip().lower().startswith("select"):
        return "Error: only SELECT queries are allowed."
    return str(db.execute(sql).fetchall())

print(query_orders("select city, sum(amount) from orders group by city"))

[('Bengaluru', 240), ('Chennai', 200)]


In [4]:
# cell 4 — describe the tool to the model (THIS dict is what an MCP server exposes for you)
tools = [{
    "name": "query_orders",
    "description": "Run a READ-ONLY SQL SELECT on the orders table (id, city, item, amount).",
    "input_schema": {
        "type": "object",
        "properties": {"sql": {"type": "string"}},
        "required": ["sql"],
    },
}]
print("Tool advertised to the model:", tools[0]["name"])

Tool advertised to the model: query_orders


In [5]:
# cell 5 — the whole MCP idea in ~10 lines: ask, run the tool, send the result back
messages = [{"role": "user", "content": "Which city has the highest total order amount? Use the tool."}]
for _ in range(5):                                   # turn limit so it can't loop forever
    reply = client.messages.create(model=MODEL, max_tokens=500, tools=tools, messages=messages)
    messages.append({"role": "assistant", "content": reply.content})
    if reply.stop_reason != "tool_use":              # model is done
        print("ANSWER:", reply.content[0].text); break
    results = []
    for block in reply.content:
        if block.type == "tool_use":
            print("Model called:", block.name, "with", block.input)
            out = query_orders(**block.input)        # YOUR code runs the tool
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": out})
    messages.append({"role": "user", "content": results})  # send result back, loop

Model called: query_orders with {'sql': 'SELECT city, SUM(amount) as total_amount FROM orders GROUP BY city ORDER BY total_amount DESC LIMIT 1'}
ANSWER: The city with the highest total order amount is **Bengaluru** with a total of **240**.


**Expected result:** the model calls `query_orders` with a `SELECT ... GROUP BY city`, you print the call, and it answers "Bengaluru" (240 vs Chennai's 200).

💡 **The punchline:** *every host in this lab — LM Studio, Claude Desktop, Claude Code — runs exactly this loop against your `server.py` automatically.* One rule to remember: **while `stop_reason == "tool_use"`, run the tool, send the result, loop again.**

**Try this:** add a second tool `top_item()` and ask "what's our best-selling item?" — watch the model pick the right tool.

🤯 **Fun fact:** MCP is often called "the tool-use loop with a standard plug". The loop existed long before MCP — MCP's contribution is agreeing on **one** way to describe the tools so you never rewrite them per app.

---

# Section 3 — Build the Server with FastMCP 🔧

Now we turn that tool into a **real MCP server** — one file, `server.py`. We use **FastMCP**, the beginner-friendly part of the official **MCP Python SDK** (`pip install mcp`). FastMCP lets you turn a normal Python function into an MCP tool with a single decorator.

🧠 **The idea:** a decorator like `@mcp.tool()` sits above a function and says *"expose this to any MCP host."* FastMCP reads your function's name, docstring, and type hints and **auto-generates the JSON schema** the host needs — you don't write the schema by hand.

### The three decorators you'll use
| Decorator | Makes a... | What it does |
|---|---|---|
| `@mcp.tool()` | **Tool** | Exposes an action the model can call |
| `@mcp.resource("uri://...")` | **Resource** | Exposes read-only data at a URI |
| `@mcp.prompt()` | **Prompt** | Exposes a reusable prompt template |

🤯 **Fun fact:** FastMCP was so useful that **FastMCP 1.0 was merged into the official MCP Python SDK** in 2024. By 2025 the SDK was being downloaded on the order of a million times a day, and some form of FastMCP powered the majority of Python MCP servers. You're learning the standard, not a side project.

## Step 1 — Prerequisites
- **Python 3.10+** with `pip` working from your terminal.
- **LM Studio 0.3.17+** (our main host) — download from https://lmstudio.ai.
- *(Optional)* **Claude Desktop** and/or **Claude Code CLI**, if you also want to test on Claude.

💻 **Sanity check** — run these in your terminal:
```bash
python --version    # 3.10+
pip install mcp     # the official MCP Python SDK (includes FastMCP)
```

## Step 2 — Create the project + seed data
Make a folder and drop in a tiny SQLite database so the server has something real to read. Run the cell below to generate the seed script, then run it once in your project folder.

In [6]:
# Pure-Python, runs anywhere — writes a seed script for a tiny SQLite orders DB.
make_db = """import sqlite3
conn = sqlite3.connect("quickcart.db")
conn.execute("DROP TABLE IF EXISTS orders")
conn.execute("CREATE TABLE orders (id INTEGER PRIMARY KEY, city TEXT, item TEXT, amount INTEGER)")
conn.executemany("INSERT INTO orders VALUES (?,?,?,?)", [
    (1, "Chennai", "milk", 120),  (2, "Chennai", "bread", 80),
    (3, "Bengaluru", "eggs", 150), (4, "Bengaluru", "milk", 90),
    (5, "Hyderabad", "bread", 60), (6, "Hyderabad", "eggs", 170),
])
conn.commit(); conn.close()
print("Created quickcart.db")
"""
with open("make_db.py", "w") as f:
    f.write(make_db)
print("✅ make_db.py written — copy it into your project folder and run: python make_db.py")

✅ make_db.py written — copy it into your project folder and run: python make_db.py


## Step 3 — Write `server.py` (tool + resource + prompt)
Run the cell to generate a complete, known-good server. Read it top to bottom — it's short, and every line is explained in the comments. Copy it into your project folder next to `quickcart.db`.

In [7]:
server_src = '''from mcp.server.fastmcp import FastMCP
import sqlite3, os

# Absolute path so the server finds the DB no matter where the host launches it from.
DB = os.path.join(os.path.dirname(__file__), "quickcart.db")

# Name your server — this is what shows up in the host's UI.
mcp = FastMCP("quickcart-orders")

# 1) A TOOL — an ACTION the model can call. Read-only guard baked into the code.
@mcp.tool()
def query_orders(sql: str) -> str:
    """Run a READ-ONLY SQL SELECT against the QuickCart orders table.
    Columns: id, city, item, amount."""
    if not sql.strip().lower().startswith("select"):
        return "Error: only SELECT queries are allowed."
    conn = sqlite3.connect(DB)
    rows = conn.execute(sql).fetchall()
    conn.close()
    return str(rows)

# 2) A RESOURCE — read-only DATA the model can pull in for context (like RAG).
@mcp.resource("orders://export/csv")
def orders_export() -> str:
    """All QuickCart orders as CSV text (id, city, item, amount)."""
    conn = sqlite3.connect(DB)
    rows = conn.execute("SELECT id, city, item, amount FROM orders").fetchall()
    conn.close()
    return "\\n".join(["id,city,item,amount"] + [",".join(map(str, r)) for r in rows])

# 3) A PROMPT — a reusable template the user can trigger in the host.
@mcp.prompt()
def daily_report() -> str:
    """A ready-made prompt that asks for today's per-city order summary."""
    return "Using the quickcart-orders tools, give me total orders and total amount per city, sorted high to low."

if __name__ == "__main__":
    mcp.run()   # defaults to the stdio transport — perfect for a local host
'''
with open("server.py", "w") as f:
    f.write(server_src)
print("✅ server.py written — copy it into your project folder next to quickcart.db")

✅ server.py written — copy it into your project folder next to quickcart.db


### 🔎 Read the server like an architect
- **The read-only guard lives in the *code*** (`if not SELECT: reject`), **not in a prompt.** This is the single most important habit today. A prompt can be talked around by a malicious instruction; a hard `if` cannot.
- **Absolute DB path** (`os.path.dirname(__file__)`) — because the *host* launches your server, and you don't control its working directory.
- **`mcp.run()` defaults to stdio** — the local transport from Section 1. Nothing leaves the machine.

🎤 **Interview angle:** *"Why put the SELECT-only check in the tool code instead of the system prompt?"* -> "Least privilege enforced where it can't be argued with. Prompt-level guards fall to prompt injection; a code-level `if not SELECT: reject` survives a poisoned document. The tool should be *incapable* of the dangerous thing, not merely *asked* not to do it."

<details><summary>✅ Quick check — which decorator makes a read-only data endpoint?</summary>

`@mcp.resource("orders://export/csv")`. `@mcp.tool()` makes an **action**; `@mcp.resource(...)` makes **read-only data**; `@mcp.prompt()` makes a **template**.
</details>

## ⚡ Optional: build it by *driving Claude Code* (the agent way)
If you have the Claude Code CLI, you don't have to hand-type the server — state the goal and let the agent build it (the loop from the Agents weekend):
```
Create a Python MCP server in server.py using FastMCP (the `mcp` package).
It manages a local SQLite quickcart.db with an orders table (id, city, item, amount).
Expose: (1) a TOOL query_orders(sql) that runs a READ-ONLY SELECT and rejects anything
that isn't a SELECT; (2) a RESOURCE orders://export/csv returning all orders as CSV;
(3) a PROMPT daily_report(). Use an absolute path to quickcart.db based on the script
location. Also create make_db.py and requirements.txt, run make_db.py, and verify server.py imports.
```
You review the output against the reference above — you're the reviewer, not the typist.

---

# 🖥️🔒 PART B — Run Your Server on a **Local Model** in LM Studio

This is the heart of the lab. You'll plug the **exact same `server.py`** into a **free, open-source model running entirely on your own laptop** — no API key, no cloud, nothing leaving your machine.

**Why an AI Architect must know this cold:**
- **Privacy / data residency** — hospital records, KYC, internal code, QuickCart's customer PII: some data legally *cannot* touch a cloud. A local model + your MCP server means the AI never phones home.
- **Cost** — after the hardware, inference is free. No per-token bill for high-volume internal tools.
- **Offline / air-gapped** — factories, defence, secure rooms with no internet.
- **Proof MCP is truly open** — the same server that talked to Claude now talks to a local Qwen model, with zero changes.

> 🧠 **Remember this:** *"Run it locally" is not a downgrade — for regulated data it's the only compliant option. MCP is what lets the same tool serve both a cloud model and a private local one.*

## B1 — Install LM Studio & download a tool-capable model

🧠 **The idea:** LM Studio is a free desktop app that (1) **runs** open-source models on your laptop and (2) is a full **MCP Host** (since **v0.3.17, June 2025**). One app = the model *and* the MCP plumbing. You do **not** need Ollama or anything else — LM Studio downloads models directly.

💻 **Steps:**
1. Download LM Studio from https://lmstudio.ai and install it.
2. Open it, go to the **Discover / search** tab, and download a **tool-capable** model. Good beginner picks that fit modest laptops and support function calling: **Qwen2.5-7B-Instruct**, **Llama-3.1-8B-Instruct**, or a small **granite** model. (Models come as **GGUF** files — LM Studio handles the format for you.)
3. Load the model in the **Chat** tab and send "hello" to confirm it runs.

> **Pick a model that supports tool use / function calling.** If a model can't call tools, it will ignore your MCP server no matter how it's configured. LM Studio flags tool-capable models — look for the wrench/tool icon.

🤯 **Fun fact:** LM Studio adopting MCP within ~7 months of its launch (Nov 2024 -> June 2025) was an early proof the standard was genuinely open. A fully *local* app choosing to speak the same protocol as Claude is exactly the vendor-neutrality Section 1 talked about.

## B2 — Where LM Studio keeps its servers: `mcp.json`

🧠 **The idea:** LM Studio stores all its MCP servers in one file called **`mcp.json`**. It follows **Cursor's `mcp.json` notation** (the same shape Cursor, Claude Desktop, and others use), so once you learn this JSON you can configure almost any host.

**Where the file lives:**
| OS | Path |
|---|---|
| **Windows** | `%USERPROFILE%\\.lmstudio\\mcp.json` |
| **macOS / Linux** | `~/.lmstudio/mcp.json` |

💻 **Easiest way to open it:** in LM Studio, go to the **Program** tab in the right sidebar -> **Install** -> **Edit mcp.json**. It opens in the built-in editor.

> ⚠️ **When pasting into an existing file:** copy only the content *inside* the `"mcpServers": { ... }` braces, so you don't create two `mcpServers` keys. LM Studio auto-loads the file when you save.

## B3 — Register your server (local stdio) 🎯

💻 Add your QuickCart server to `mcp.json`. Use an **absolute path** to `server.py`:

**Windows:**
```json
{
  "mcpServers": {
    "quickcart-orders": {
      "command": "python",
      "args": ["C:\\Users\\you\\quickcart-mcp\\server.py"]
    }
  }
}
```

**macOS / Linux:**
```json
{
  "mcpServers": {
    "quickcart-orders": {
      "command": "python",
      "args": ["/Users/you/quickcart-mcp/server.py"]
    }
  }
}
```

Save the file. LM Studio spawns your server in **its own separate process** (isolation — nice for safety). Now load your tool-capable model, open a chat, and ask:
```
Using quickcart-orders, which city has the highest total order amount?
```

**What you'll see:** LM Studio pops a **tool-call confirmation dialog** showing the exact tool + arguments *before* it runs. Approve it -> the local model reads your database and answers. **No key. No cloud. Same `server.py` you'd use on Claude.** 🎉

> 🔒 That confirmation dialog is a **feature, not a nuisance** — it's your human-in-the-loop guardrail. You can "always allow" a trusted tool, but *you* decide. Each server also runs in its own process, so a misbehaving server can't reach into LM Studio itself.

🎤 **Interview angle:** *"How did you connect a local model to a database safely?"* -> "An MCP server with a read-only, SELECT-only tool, registered in LM Studio's `mcp.json` over stdio. LM Studio runs it in an isolated process and gates every tool call behind a human approval dialog. The model never touches the DB directly — only the narrow tool I exposed."

## B4 — The `python` gotcha (read before you rage-quit) 🛠️

The #1 reason `mcp.json` "doesn't work" is that LM Studio can't find `python`. LM Studio may not share your terminal's PATH.

**Fixes, in order:**
1. Use the **full path** to your Python in `"command"`, e.g. `"C:\\Users\\you\\AppData\\Local\\Programs\\Python\\Python312\\python.exe"` or `/usr/bin/python3`. Find it with `where python` (Windows) or `which python3` (macOS/Linux).
2. Make sure the **`mcp` package is installed in that same Python** (`<full-python> -m pip install mcp`).
3. Use an **absolute path** to `server.py` and to `quickcart.db` (the server already does the latter).

<details><summary>✅ Quick check 1 — how many lines of server.py did you change to run on a local model?</summary>

**Zero.** Different host, different `mcp.json` — identical server. That's the open standard doing its job: build once, run anywhere.
</details>

## B5 — Drive MCP from LM Studio's **API** (your code, a local model, real tools) 💻

The chat UI is great for demos. For a real product you want your *code* to call a local model that can use MCP tools. LM Studio exposes an API for exactly this (**requires LM Studio 0.4.0+**).

🧠 **The idea:** LM Studio runs a local server (default `http://localhost:1234`). You send a chat request and attach **integrations** — either an **ephemeral** MCP server (defined in the request, great for remote servers) or a **pre-configured** one from your `mcp.json` (great for local command servers). You can restrict which tools the model may call with **`allowed_tools`** — that's tool budgeting (Section on context) enforced by the API itself.

💻 First, in LM Studio -> **Developer** tab, **start the server** and (in Server Settings) enable **"Allow calling servers from mcp.json"** (and "Allow per-request MCPs" if you'll use ephemeral servers).

In [ ]:
# Call a LOCAL model + your mcp.json server, from Python. No Anthropic key — this hits LM Studio.
# Requires: LM Studio 0.4.0+, its server running, and quickcart-orders in mcp.json.
import requests, os

LM_URL   = "http://localhost:1234/api/v1/chat"
LM_TOKEN = os.environ.get("LM_API_TOKEN", "lm-studio")  # LM Studio's local token

payload = {
    "model": "qwen2.5-7b-instruct",          # whatever tool-capable model you loaded
    "input": "Which city has the highest total order amount?",
    "integrations": [
        {                                    # pull the pre-configured server from mcp.json
            "type": "plugin",
            "id": "mcp/quickcart-orders",
            "allowed_tools": ["query_orders"]  # tool budgeting: only this tool is offered
        }
    ],
    "context_length": 8000,
    "temperature": 0
}
r = requests.post(LM_URL, headers={"Authorization": f"Bearer {LM_TOKEN}"}, json=payload)
print(r.json())

**Expected result:** the JSON response contains a `"type": "tool_call"` block (the local model calling `query_orders`) followed by a `"type": "message"` block with the answer — the **same tool-use loop** as the Claude cell in Section 2, but running on a *local* model over MCP.

> 💡 **The two ways to attach a server (memorize this table):**
>
> | | Ephemeral | From `mcp.json` |
> |---|---|---|
> | `type` | `"ephemeral_mcp"` | `"plugin"` |
> | Defined | Per-request (great for **remote** servers) | Pre-configured (great for **local `command`** servers) |
> | Server id | `server_label` + `server_url` | `id` (e.g. `quickcart-orders`) |
> | Limit tools | `allowed_tools` | `allowed_tools` |

🎤 **Interview angle:** *"How would you limit what a local agent can do at the API layer?"* -> "`allowed_tools` on the integration. Even if the server exposes ten tools, I pass only the ones this task needs — fewer tokens *and* a smaller attack surface. Budgeting isn't just a UI toggle; it's a request parameter."

---

# Section 4 — Same Server on Claude (2 minutes, to prove "run anywhere") 🔁

You've done the hard, valuable part (local + LM Studio). Here's the *quick* proof that the identical `server.py` also runs on Claude — zero code changes, only config.

**Claude Desktop** — Settings -> Developer -> **Edit Config** opens `claude_desktop_config.json`:
- Windows: `%APPDATA%\\Claude\\claude_desktop_config.json`
- macOS: `~/Library/Application Support/Claude/claude_desktop_config.json`

```json
{
  "mcpServers": {
    "quickcart-orders": {
      "command": "python",
      "args": ["/abs/path/quickcart-mcp/server.py"]
    }
  }
}
```
Notice it's the **same shape** as LM Studio's `mcp.json`. **Fully quit and reopen** Claude Desktop, then ask *"Using quickcart-orders, which city has the highest total?"*

**Claude Code CLI** — one command, or a shareable project file:
```bash
claude mcp add quickcart-orders -- python /abs/path/quickcart-mcp/server.py
claude mcp list          # confirms it's registered
```
Or commit a `.mcp.json` in the project root (same JSON) so your whole team gets it.

> 🧠 **Remember this:** *LM Studio's `mcp.json`, Claude Desktop's `claude_desktop_config.json`, and Claude Code's `.mcp.json` are the **same `mcpServers` shape**. Learn it once, configure any host.*

🎤 **Interview angle:** *"A teammate says 'just hardcode the DB call into each app.' Why is MCP better?"* -> "Three apps × one change = three edits and three redeploys, and the safety guard gets copy-pasted three times where one copy will drift. With MCP it's one server edit; every host — local or cloud — picks it up. One guard, one place."

---

# 🏗️ Architecture — The Whole System, One Picture

Here's everything you built, drawn the way an architect would for a design review. Notice the **local model** is the primary path for sensitive data.

```
        HUMAN (QuickCart ops teammate)
                 │  natural-language question
   ┌─────────────┼──────────────────────────────┐
   ▼             ▼                                ▼
┌───────────┐ ┌──────────┐                 ┌──────────┐
│ LM STUDIO │ │ Claude   │  ... any host   │  future  │
│ local Qwen│ │ Desktop  │      speaking   │  host    │
│ (PII data)│ │ (general)│      MCP        │          │
└─────┬─────┘ └────┬─────┘                 └────┬─────┘
      │  MCP client (one per server, inside each host)
      └────────────┴──────────────┬───────────────┘
                                  ▼  stdio (JSON-RPC 2.0)
                     ┌────────────────────────────┐
                     │  quickcart-orders  SERVER   │  <- isolated process
                     │  guard: SELECT-only  ✅      │  <- least privilege
                     │  tool: query_orders          │  <- human-approved calls
                     │  resource: orders://csv      │  <- audit log (add for prod)
                     │  prompt: daily_report        │
                     └──────────────┬──────────────┘
                                    ▼  read-only
                          ┌──────────────────┐
                          │  quickcart.db     │  (never leaves the machine)
                          └──────────────────┘
```

| Component | Responsibility | Main failure point | Guardrail |
|---|---|---|---|
| **Host** | Talk to human, decide when to call a tool | Over-eager / injected instructions | Human approval dialog |
| **Client** | Bridge host ↔ one server | Server not launched / bad path / `python` not found | absolute paths, full python path |
| **Server** | Expose narrow, safe capabilities | Over-broad tool (write, shell) | Least privilege, validation |
| **Data** | Hold the real records | Unrestricted writes/reads | Read-only, scoped queries |

**Scaling / security / cost tradeoffs an architect weighs:**
- **Scaling:** one server, many hosts (**N+M**) beats bespoke integrations (**N×M**). Stateless stdio servers are easy to replicate.
- **Security:** every tool is attack surface. Read-only + allowlist + human-in-loop + sandbox + logs. Treat all tool inputs as untrusted (prompt injection).
- **Cost & privacy:** **local model** = free inference + data stays home, but tight context (budget your tools). **Cloud model** = per-token bill but huge context. Choose per data-sensitivity and volume — QuickCart routes PII to the local model.

# 🚀 Mini Project — "QuickCart Ops Copilot" (privacy-first, local-first) 🖥️

**Business case:** QuickCart's ops team wants a chat copilot for order questions. Customer data is sensitive, so it **must run on a local model in LM Studio** — nothing leaves the building — while general staff *may* use Claude Desktop for non-sensitive questions. **One server must serve both.**

**Architecture:** the `quickcart-orders` MCP server -> **LM Studio + local model** (PII queries, primary) *and* **Claude Desktop** (general). Read-only tools + one guarded write (`mark_refunded`) behind human approval.

**Build steps (local-first):**
1. Extend `server.py` with `top_city()`, `top_item()`, and a **read-only** `city_report(city)` tool. Keep the SELECT-only guard.
2. Add a **write** tool `mark_refunded(order_id, approved=False)` that (a) validates the id exists, (b) returns a "needs approval" message unless `approved=True` — your human-in-the-loop gate.
3. Add **audit logging**: append every tool call + args to `audit.log`.
4. Register the server in **LM Studio's `mcp.json`** (primary) and **Claude Desktop** (secondary). Confirm the *same* server answers in both.
5. In LM Studio, set the context window sensibly and use **`allowed_tools`** (or the UI) to enable **only** the read tools by default.
6. Ask the same question in both hosts; confirm identical answers and that `mark_refunded` refuses without approval.

**Definition of done:**
- ✅ One `server.py` runs on a **local model in LM Studio** *and* Claude Desktop, unchanged.
- ✅ Read queries work; `mark_refunded` blocks until a human approves.
- ✅ `audit.log` shows every call.
- ✅ You can explain, in one minute, why the sensitive queries stay on the local model.

**Stretch:** call it from **LM Studio's API** (B5) with `allowed_tools`; add scope validation that rejects any query naming a table other than `orders`; containerize the server.

---

# 📚 Revision Suite

## Session Summary
MCP is a **USB-C port for AI apps**: build a capability **once** as a server, plug it into **any** host. Under the metaphor it's plain **JSON-RPC 2.0** over two transports (**stdio** for local, **Streamable HTTP** for remote), exposing up to **six primitives** — server-side **tools / resources / prompts**, client-side **sampling / roots / elicitation**. Today you built a read-only QuickCart orders server with **FastMCP** (tool + resource + prompt), then ran it on a **local open-source model in LM Studio** — no key, no cloud — including from **LM Studio's API** with `allowed_tools`, and proved the *same* server runs on Claude unchanged. You learned to **budget context and tools** so local + MCP doesn't crash, and studied the **HexStrike** incident to see why narrow tools + humans-in-the-loop are the real job.

## ⏱️ 5-Minute Revision Guide
1. **MCP = USB-C for AI.** Build once (server), plug into any host. Underneath: **JSON-RPC 2.0**.
2. **6 primitives:** tools (run), resources (read), prompts (template) + sampling, roots, elicitation.
3. **2 transports:** **stdio** (local) vs **Streamable HTTP** (remote).
4. **Governance:** Anthropic made it (Nov 2024), gave it to the **Linux Foundation** (Dec 2025). Vendor-neutral.
5. **N+M > N×M** — the whole business case.
6. **Same server, every host** — LM Studio, Claude Desktop, Claude Code — only *config* differs. It's the same `mcpServers` JSON.
7. **Local models** = privacy + free inference + offline, at the cost of **tight context**.
8. **Tools pay token rent** every turn -> budget them (`allowed_tools`); raise the window if VRAM allows.
9. **Safety isn't in the protocol, it's in your tools:** read-only, allowlist, human-in-loop, sandbox, logs.

## 🎤 Interview Preparation Notes
- **"What is MCP, simply?"** A USB-C port for AI — one open standard so any AI app can use any tool via a server, no custom glue per pairing. Under the hood it's JSON-RPC 2.0.
- **"Name the primitives."** Server-side: tools (actions), resources (read-only data), prompts (templates). Client-side: sampling (server borrows the host's model), roots (allowed folders), elicitation (server asks the user).
- **"stdio vs Streamable HTTP?"** stdio = a local process over stdin/stdout, for servers on your machine. Streamable HTTP = a remote web service (replaced the old HTTP+SSE). Local data -> stdio.
- **"Is MCP a Claude feature / lock-in?"** No — Anthropic open-sourced it (Nov 2024) and *donated it to the Linux Foundation* (Dec 2025). LM Studio, Cursor, OpenAI all use it.
- **"Why run MCP on a local model?"** Data can't leave the building (PII/KYC/health), no per-token cost, works offline.
- **"Why does local + MCP crash?"** Tool schemas overflow a small context window — budget tools (`allowed_tools`), raise the window, or pick a bigger-context model.
- **(Architecture) "Design a safe ops agent."** Read-only by default, writes behind human approval, allowlist per task, sandboxed servers, audit logs, inputs treated as hostile.
- **(FDE) "A client fears the AI will 'go rogue' with tools."** Show LM Studio's approval dialog and the SELECT-only guard live; explain least privilege and cite HexStrike as why human-in-the-loop is the standard.

## 📝 Assignment
- **Beginner:** Register `quickcart-orders` in **LM Studio's `mcp.json`**, load a tool-capable model, and ask two questions — one that hits the tool, one that reads the CSV resource. Screenshot the approval dialog.
- **Intermediate:** Call the *same* server from **LM Studio's API** (B5) with `allowed_tools=["query_orders"]`. Show the `tool_call` block in the JSON response.
- **Advanced:** Measure the token cost of your tool list (Section 4 cell). Add a 4th and 5th tool; show how the budget grows and decide what to keep enabled by default.
- **Project:** Complete the **QuickCart Ops Copilot** with the guarded `mark_refunded` write tool, audit logging, and a one-paragraph threat model referencing least privilege + prompt injection.

## 🧪 Assessment

### Part 1 — Multiple Choice (10)
1. In MCP, the **host** is: (a) your server file (b) the AI app the human uses (c) the database (d) the tool schema
2. A **resource** in MCP is best described as: (a) an action the AI performs (b) read-only data the AI can pull in (c) a database password (d) a host
3. The wire format MCP is built on is: (a) GraphQL (b) JSON-RPC 2.0 (c) gRPC (d) SOAP
4. For a server running **locally** on your laptop, the right transport is: (a) Streamable HTTP (b) stdio (c) WebRTC (d) FTP
5. Which is a **client-side** primitive: (a) tool (b) resource (c) sampling (d) prompt
6. As of December 2025, MCP is governed by: (a) Anthropic alone (b) the Linux Foundation (AAIF) (c) OpenAI (d) no one
7. LM Studio became an **MCP Host** in: (a) it never did (b) v0.3.17, June 2025 (c) 2023 (d) only for cloud models
8. LM Studio's `mcp.json` for a **remote** server uses: (a) `command` + `args` (b) `url` + `headers` (c) `model` (d) `pip`
9. On a laptop, connecting a huge MCP server often causes: (a) faster answers (b) context overflow / crashes (c) lower cost (d) better safety
10. The safest place to enforce "SELECT only" is: (a) the system prompt (b) the tool's code (c) the model's training (d) the user's manners

### Part 2 — Short Answer (5)
1. Explain the MCP tool-use loop in one sentence.
2. Name the three server-side primitives and what each is for.
3. Give two concrete reasons an enterprise would run MCP on a **local** model in LM Studio.
4. Why do tool definitions "pay rent" in a context window, and why does that hurt local models more?
5. Why is a code-level guard (`if not SELECT: reject`) safer than a prompt instruction?

### Part 3 — Scenario (3)
1. Your local model in LM Studio ignores your server entirely — no tool call, no error. Give two likely causes and fixes.
2. A client wants an AI that can "just run any command on our servers to fix things." As the architect, what do you propose instead, and why?
3. Legal forbids sending customer PII to any cloud, but staff want AI over that data. Sketch the architecture and name the one server that serves both your local and cloud hosts.

## ✅ Answer Key

**MCQ:** 1-b · 2-b · 3-b · 4-b · 5-c · 6-b · 7-b · 8-b · 9-b · 10-b

**Short answer:**
1. While the model's `stop_reason` is `tool_use`, run the requested tool, send the result back, and loop until it stops.
2. **Tool** = an action the model calls (it runs code); **Resource** = read-only data the model pulls in for context; **Prompt** = a reusable prompt template the server offers.
3. Data can't leave the building (PII/KYC/health/compliance); no per-token cost; also works offline/air-gapped (any two).
4. Every tool's name + description + schema is text the model must hold in context *every turn*; local windows are small (4k–8k), so a big tool list leaves no room to reason and can crash the machine.
5. Prompts can be talked around (prompt injection); code runs regardless of what the model was told. The guard belongs where it can't be argued with.

**Scenario:**
1. Likely: (a) the loaded model **can't do function calling** -> switch to a tool-capable model (Qwen 2.5 / Llama 3.1); (b) LM Studio **can't find `python`** or the path to `server.py`/`mcp` package is wrong -> use the full python path and absolute paths, install `mcp` in that python. (Also possible: context overflow from too many tools.)
2. Reject the general "run any command" tool (unbounded power + prompt-injection risk). Propose narrow, validated tools per task, read-only by default, writes behind human approval, sandboxed and logged — least privilege over a blank cheque.
3. A **local model in LM Studio** as the host for PII queries + the same `quickcart-orders`-style MCP server (stdio, read-only); general staff use Claude Desktop with the **same server**. One server, two hosts, PII never leaves the building.

---

## 🛠️ Troubleshooting
| Problem | Cause | Fix |
|---|---|---|
| LM Studio doesn't see the server | `mcp.json` not saved / bad path | Save the file (auto-loads); use **absolute** path to `server.py` |
| `spawn python ENOENT` | `python` not on LM Studio's PATH | Put the **full path** to python in `"command"` (`where python` / `which python3`) |
| `ModuleNotFoundError: mcp` | SDK not in the python LM Studio launches | `"<full-python>" -m pip install mcp` |
| Local model ignores tools | Model can't do function calling **or** context overflow | Use a tool-capable model (Qwen 2.5 / Llama 3.1); trim tools; raise context |
| `no such table: orders` | DB not built / wrong path | Run `python make_db.py`; server already uses an absolute DB path |
| Model crashes / machine freezes | Model + context too big for RAM/VRAM | Smaller/quantized (q4) model, CPU offload, shorter context |
| Tool call denied | Approval prompt dismissed | Re-try and **approve** (that prompt is your safety net) |
| API `/api/v1/chat` 403 / ignores server | Per-request / mcp.json calls disabled | In Server Settings enable "Allow calling servers from mcp.json" (and per-request MCPs) |
| Claude Desktop server not showing | Config not reloaded | Fully **quit** and restart Claude Desktop |

## 🔍 Resources (Official & Verified)
- MCP spec & docs — https://modelcontextprotocol.io  ·  spec version 2025-11-25
- Transports (stdio / Streamable HTTP) — https://modelcontextprotocol.io/specification/2025-03-26/basic/transports
- MCP Python SDK (FastMCP) — https://github.com/modelcontextprotocol/python-sdk
- LM Studio — Use MCP Servers — https://lmstudio.ai/docs/app/mcp
- LM Studio — Using MCP via API — https://lmstudio.ai/docs/developer/core/mcp
- LM Studio 0.3.17 (MCP Host) blog — https://lmstudio.ai/blog/lmstudio-v0.3.17
- Anthropic — donating MCP to the Agentic AI Foundation — https://www.anthropic.com/news/donating-the-model-context-protocol-and-establishing-of-the-agentic-ai-foundation
- Connect Claude Code / Desktop to MCP — https://docs.claude.com/en/docs/claude-code/mcp
- *Legal* practice grounds — PortSwigger Web Security Academy · TryHackMe · Hack The Box

## 🎁 Wrap-up
**What you did:** built one MCP server with FastMCP (tool + resource + prompt), ran it live on a **local open-source model in LM Studio** — chat UI *and* API — connected a remote server too, and proved the *same* server runs on Claude unchanged. You learned the protocol under the metaphor (JSON-RPC, primitives, transports, governance), how to budget context and tools so local + MCP doesn't crash, and the safety architecture a real incident (HexStrike) proves you can't skip.

**The one line to carry forward:** *MCP makes a capability reusable on any brain — cloud or local; your job as an architect is to expose the smallest, safest tools that get the job done, with a human on the trigger of anything that writes.*

✅ **Lab complete.**